# Hisab Pata — Gemma 3 1B Fine-tune on Kaggle

Model: `google/gemma-3-1b-pt` (base, text-only)
Dataset: 6832 Bengali/Banglish examples
Method: QLoRA (4-bit) via SFTTrainer

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [ ]:
# ── CHANGE THIS PATH ──
# Option A: Upload train.jsonl directly here
DATA_PATH = '/kaggle/input/hisab-ai-dataset/train.jsonl'  # Kaggle Dataset mount path

import json
with open(DATA_PATH) as f:
    data = [json.loads(l) for l in f]
print(f'Loaded {len(data)} examples')

In [ ]:
from datasets import Dataset

def format_chat(ex):
    """Convert ChatML → Gemma 3 chat format for SFT"""
    # Gemma 3 format:
    # <bos><start_of_turn>user\nsystem prompt\n\nuser message<end_of_turn>\n<start_of_turn>model\nresponse<end_of_turn>
    msgs = ex['messages']
    # System prompt becomes prefix to first user message
    sys_content = ''
    remaining = msgs
    if msgs and msgs[0]['role'] == 'system':
        sys_content = msgs[0]['content']
        remaining = msgs[1:]
    parts = []
    for i, m in enumerate(remaining):
        role = 'model' if m['role'] == 'assistant' else 'user'
        content = m['content']
        if i == 0 and sys_content:
            content = sys_content + '\n\n' + content
        parts.append(f"<start_of_turn>{role}\n{content}<end_of_turn>")
    return '<bos>' + '\n'.join(parts)

texts = [format_chat(ex) for ex in data]
dataset = Dataset.from_list([{'text': t} for t in texts])
print(f'Format example:\n{texts[0][:400]}...')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

MODEL_NAME = 'google/gemma-3-1b-pt'  # BASE model (pretrained, text-only)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print('Model loaded ✓')

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir='/kaggle/working/hisabpai',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy='epoch',
        report_to='none',
        bf16=True,
        max_grad_norm=0.3,
    ),
    tokenizer=tokenizer,
    formatting_func=lambda x: x['text'],
    max_seq_length=512,
    peft_config=lora_config,
)

print('Starting training...')
trainer.train()
print('Training complete ✓')

In [ ]:
# Save LoRA adapters
OUT = '/kaggle/working/hisabpai-final'
trainer.model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print(f'Saved to {OUT}')

# Zip for download
!zip -r /kaggle/working/hisabpai-gemma3-1b.zip {OUT}
print('ZIP ready for download')

In [ ]:
# Quick test
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map='auto', torch_dtype=torch.bfloat16
)
model_lora = PeftModel.from_pretrained(base, OUT)

tests = [
    '500 টাকা খরচ করেছি পরিবহন বাবদ',
    'কে তুমি?',
    'ব্যালেন্স কত?',
    'রহিম কে ২০০ টাকা সেন্ড করেছি',
]

for inp in tests:
    prompt = '<bos><start_of_turn>user\nbook_type: personal\ncategories: Transport, Mobile Recharge, Postage, Publication, Office Stationery, Tips, Donation, Others, Salary\nbalance: 12500\n\n' + inp + '<end_of_turn>\n<start_of_turn>model\n'
    tokens = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model_lora.generate(**tokens, max_new_tokens=128)
    resp = tokenizer.decode(out[0], skip_special_tokens=True)
    if '<start_of_turn>model' in resp:
        resp = resp.split('<start_of_turn>model')[-1].strip()
    print(f'\nQ: {inp}')
    print(f'A: {resp[:200]}')